In [ ]:
# import de la librería pandas
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.diagnostic as sms
from statsmodels.tsa.stattools import adfuller
from dotenv import load_dotenv
import os

# importar ruta desde un .env file
load_dotenv()
ruta_absoluta = os.getenv('RUTA_ABSOLUTA')

# cargar datos de un archivo excel
df = pd.read_excel(f"{ruta_absoluta}/data/processed/Base_datos.xlsx")


# Modelo de regresiòn lineal multiple
y = df["CR_diff"]
x = df[['TIR', 'TM', 'TD', 'NO']]

# agregar contante
X = sm.add_constant(x)

# modelar OLS
modelo = sm.OLS(y, X).fit()

residuos = modelo.resid
# resumen del modelo
print(modelo.summary())


# Prueba más robusta: Breusch-Godfrey (para autocorrelación de orden superior, ej. 2 rezagos)
bg_test = sms.acorr_breusch_godfrey(modelo, nlags=2)
bg_pvalue = bg_test[1]
print(f"p-valor (Breusch-Godfrey): {bg_pvalue:.4f}")
if bg_pvalue > 0.05:
    print("Resultado (BG): No hay autocorrelación en los residuos.")
else:
    print("Resultado (BG): Se rechaza H0. Existe AUTOCORRELACIÓN.")
print("\n" + "="*60 + "\n")

print("======= 5. HETEROCEDASTICIDAD (White) =======")
bp_test = sms.het_white(residuos, X)

bp_pvalue = bp_test[1]
print(f"p-valor (White): {bp_pvalue:.4f}")
if bp_pvalue > 0.05:
    print("Resultado: No se rechaza H0. Los residuos son HOMOCEDÁSTICOS (Varianza constante).")
else:
    print("Resultado: Se rechaza H0. Los residuos sufren de HETEROCEDASTICIDAD.")



                            OLS Regression Results                            
Dep. Variable:                CR_diff   R-squared:                       0.126
Model:                            OLS   Adj. R-squared:                  0.098
Method:                 Least Squares   F-statistic:                     4.544
Date:                Sat, 13 Jun 2026   Prob (F-statistic):            0.00184
Time:                        22:23:37   Log-Likelihood:                 412.71
No. Observations:                 131   AIC:                            -815.4
Df Residuals:                     126   BIC:                            -801.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0130      0.025      0.513      0.6